In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import os
os.chdir('..')
os.getcwd()


## Linear regression

In [2]:
import autograd.numpy as anp
import numpy as np
from sazz.samplers.AutomaticBoomerang import AutomaticBoomerangSampler
from sazz.samplers.AutomaticBoomerang import FactorizedAutomaticBoomerangSampler
from sazz.samplers.AutomaticBoomerang import StickyAutomaticBoomerangSampler
from sazz.samplers.AutomaticBoomerang import FactorizedStickyAutomaticBoomerangSampler

In [3]:
# ============================================================
# Simulate data
# ============================================================

def simulate_linear_regression(n=200, p=5, sigma=0.5, seed=1):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, p))
    beta_true = np.array([2.0, -1.5, 0.0, 1.0, 0.5])[:p]
    if len(beta_true) < p:
        beta_true = np.concatenate([beta_true, np.zeros(p - len(beta_true))])
    y = X @ beta_true + rng.normal(scale=sigma, size=n)
    return X, y, beta_true, sigma


def make_E_linear_regression(X, y, sigma=0.5, tau=2.0):
    """
    Bayesian linear regression with:
        y | beta ~ N(X beta, sigma^2 I)
        beta ~ N(0, tau^2 I)

    Returns E(beta) = negative log posterior up to additive constant.
    """
    X = anp.array(X)
    y = anp.array(y)

    def E(beta):
        resid = y - X @ beta
        nll = 0.5 / sigma**2 * anp.sum(resid**2)
        nprior = 0.5 / tau**2 * anp.sum(beta**2)
        return nll + nprior

    return E


In [4]:
# Linear regression
X_lin, y_lin, beta_true_lin, sigma_true = simulate_linear_regression(n=200, p=10, sigma=0.5, seed=1)
E_lin = make_E_linear_regression(X_lin, y_lin, sigma=sigma_true, tau=2.0)

In [5]:
def make_gradE_linear_regression(X, y, sigma=0.5, tau=2.0):
    X = np.asarray(X)
    y = np.asarray(y)

    def gradE(beta):
        resid = y - X @ beta
        return -(X.T @ resid) / sigma**2 + beta / tau**2

    return gradE

def make_partial_gradE_linear_regression(X, y, sigma=0.5, tau=2.0):
    X = np.asarray(X)
    y = np.asarray(y)

    def partial_gradE(beta, i):
        resid = y - X @ beta
        return -(X[:, i] @ resid) / sigma**2 + beta[i] / tau**2

    return partial_gradE

gradE = make_gradE_linear_regression(X_lin, y_lin, sigma=sigma_true)
partial_gradE = make_partial_gradE_linear_regression(X_lin, y_lin, sigma=sigma_true)


In [6]:
sampler_lin = AutomaticBoomerangSampler(E_lin, dim=X_lin.shape[1], gradE=gradE)
sampler_partial_lin = FactorizedAutomaticBoomerangSampler(E_lin, dim=X_lin.shape[1], gradE=gradE, partial_gradE=partial_gradE)
sampler_sticky_lin = StickyAutomaticBoomerangSampler(E_lin, dim=X_lin.shape[1], kappa=1, gradE=gradE)
sampler_partial_sticky_lin = FactorizedStickyAutomaticBoomerangSampler(E_lin, dim=X_lin.shape[1], kappa=1, gradE=gradE, partial_gradE=partial_gradE)

In [7]:
import numpy as np

# x_ref_right = np.array([1.0, 0.0])
# x_ref_left = np.array([-1.0, 0.0])
# Sigma_inv_well = np.array([
#     [2.0, 0.0],
#     [0.0, 1.0]
# ])

sampler_lin.preprocess(
    method="laplace",
    # x_ref=x_ref_right,
    # Sigma_inv=Sigma_inv_well
)

sampler_partial_lin.preprocess(
    method="diagonal",
    # x_ref=x_ref_right,
    # Sigma_inv=Sigma_inv_well
)

sampler_sticky_lin.preprocess(
    method="laplace",
    # x_ref=x_ref_right,
    # Sigma_inv=Sigma_inv_well
)

sampler_partial_sticky_lin.preprocess(
    method="diagonal",
    # x_ref=x_ref_right,
    # Sigma_inv=Sigma_inv_well
)

In [ ]:
linreg_auto_bs = sampler_lin.sample_auto(T=2000, adapt_t_max=True, refine=True, safety=1.10, n_grid=10)
linreg_t_auto_bs = linreg_auto_bs["t"]
linreg_x_auto_bs = linreg_auto_bs["x"]

In [ ]:
linreg_out_auto_fbs = sampler_partial_lin.sample_auto(T=2000, adapt_t_max=True, refine=True, safety=1.10, n_grid=10)
linreg_t_auto_fbs = linreg_out_auto_fbs["t"]
linreg_x_auto_fbs = linreg_out_auto_fbs["x"]

In [ ]:
linreg_out_auto_sticky_bs = sampler_sticky_lin.sample_auto(T=2000, adapt_t_max=True, refine=True, safety=1.10, n_grid=10)
linreg_t_auto_sticky_bs = linreg_out_auto_sticky_bs["t"]
linreg_x_auto_sticky_bs = linreg_out_auto_sticky_bs["x"]

In [ ]:
linreg_out_auto_partial_sticky_bs = sampler_partial_sticky_lin.sample_auto(T=2000, adapt_t_max=True, refine=True, safety=1.10, n_grid=10)
linreg_t_auto_sticky_bs = linreg_out_auto_partial_sticky_bs["t"]
linreg_x_auto_sticky_bs = linreg_out_auto_partial_sticky_bs["x"]

In [12]:
import numpy as np
import matplotlib.pyplot as plt

def resample_pdmp_path(output, sampler, n_samples=4000, sticky=False):
    t_sk = output["t"]
    x_sk = output["x"]
    v_sk = output["v"]

    t_grid = np.linspace(0.0, t_sk[-1], n_samples)

    # segment index for each resample time
    idx = np.searchsorted(t_sk, t_grid, side="right") - 1
    idx = np.clip(idx, 0, len(t_sk) - 2)

    x_res = np.empty((n_samples, x_sk.shape[1]))

    if sticky:
        frozen_sk = output["frozen_mask"]
        frozen_res = frozen_sk[idx]   # propagate mask along segments

    for j, tg in enumerate(t_grid):
        i = idx[j]
        dt = tg - t_sk[i]

        if sticky:
            x_res[j], _ = sampler._state_on_orbit_sticky(
                x_sk[i], v_sk[i], frozen_sk[i], dt
            )
        else:
            x_res[j], _ = sampler.state_on_orbit(
                x_sk[i], v_sk[i], dt
            )

    if sticky:
        return t_grid, x_res, frozen_res
    else:
        return t_grid, x_res


# ------------------------------------------------------------
# Uniform-in-time samples
# ------------------------------------------------------------
t_bs, samp_bs = resample_pdmp_path(linreg_auto_bs, sampler_lin, n_samples=4000, sticky=False)
t_fbs, samp_fbs = resample_pdmp_path(linreg_out_auto_fbs, sampler_partial_lin, n_samples=4000, sticky=False)
t_sbs, samp_sbs, frozen_sbs = resample_pdmp_path(linreg_out_auto_sticky_bs, sampler_sticky_lin, n_samples=4000, sticky=True)
t_fsbs, samp_fsbs, frozen_fsbs = resample_pdmp_path(linreg_out_auto_partial_sticky_bs, sampler_partial_sticky_lin, n_samples=4000, sticky=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1) Posterior summaries
# ------------------------------------------------------------
def posterior_summary(samples):
    mean = np.mean(samples, axis=0)
    q05 = np.quantile(samples, 0.05, axis=0)
    q95 = np.quantile(samples, 0.95, axis=0)
    return mean, q05, q95

mean_bs, q05_bs, q95_bs = posterior_summary(samp_bs)
mean_fbs, q05_fbs, q95_fbs = posterior_summary(samp_fbs)
mean_sbs, q05_sbs, q95_sbs = posterior_summary(samp_sbs)
mean_fsbs, q05_fsbs, q95_fsbs = posterior_summary(samp_fsbs)

p = samp_bs.shape[1]
idx = np.arange(p)

plt.figure(figsize=(10, 5))

plt.errorbar(idx - 0.3, mean_bs, 
             yerr=[mean_bs - q05_bs, q95_bs - mean_bs],
             fmt='o', capsize=4, label='Boomerang')

plt.errorbar(idx - 0.1, mean_fbs, 
             yerr=[mean_fbs - q05_fbs, q95_fbs - mean_fbs],
             fmt='o', capsize=4, label='Factorized')

plt.errorbar(idx + 0.1, mean_sbs, 
             yerr=np.maximum(0.0, np.min([mean_sbs - q05_sbs, q95_sbs - mean_sbs])),
             fmt='o', capsize=4, label='Sticky')

plt.errorbar(idx + 0.3, mean_fsbs, 
             yerr=np.maximum(0.0, np.min([mean_fsbs - q05_fsbs, q95_fsbs - mean_fsbs])),
             fmt='o', capsize=4, label='Factorized Sticky')

plt.scatter(idx, beta_true_lin, marker='x', s=80, color='black', label='True')
plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.xticks(idx, [f"$\\beta_{j}$" for j in range(p)])
plt.ylabel("Coefficient value")
plt.title("Posterior means and 90% intervals")
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 2) Trace + histogram for selected coefficients
# ------------------------------------------------------------
sel = np.arange(min(3, p))   # first 3 coefficients

fig, axes = plt.subplots(len(sel), 2, figsize=(12, 3 * len(sel)))
if len(sel) == 1:
    axes = np.array([axes])

for r, k in enumerate(sel):
    # trace
    axes[r, 0].plot(t_bs, samp_bs[:, k], lw=1, alpha=0.8, label="Boomerang")
    axes[r, 0].plot(t_fbs, samp_fbs[:, k], lw=1, alpha=0.8, label="Factorized")
    axes[r, 0].plot(t_sbs, samp_sbs[:, k], lw=1, alpha=0.8, label="Sticky")
    axes[r, 0].plot(t_fsbs, samp_fsbs[:, k], lw=1, alpha=0.8, label="Factorized Sticky")
    axes[r, 0].axhline(beta_true_lin[k], color="black", linestyle="--", alpha=0.7)
    axes[r, 0].set_title(f"Trace: beta[{k}]")
    axes[r, 0].set_xlabel("t")
    axes[r, 0].set_ylabel(f"beta[{k}]")

    # histogram
    axes[r, 1].hist(samp_bs[:, k], bins=40, density=True, alpha=0.4, label="Boomerang")
    axes[r, 1].hist(samp_fbs[:, k], bins=40, density=True, alpha=0.4, label="Factorized")
    axes[r, 1].hist(samp_sbs[:, k], bins=40, density=True, alpha=0.4, label="Sticky")
    axes[r, 1].hist(samp_fsbs[:, k], bins=40, density=True, alpha=0.4, label="Factorized Sticky")
    axes[r, 1].axvline(beta_true_lin[k], color="black", linestyle="--", alpha=0.7)
    axes[r, 1].set_title(f"Marginal: beta[{k}]")
    axes[r, 1].set_xlabel(f"beta[{k}]")

axes[0, 0].legend()
axes[0, 1].legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 3) Sticky-specific: fraction of time frozen
# ------------------------------------------------------------
frozen_frac = np.mean(frozen_sbs, axis=0)
partial_frozen_frac = np.mean(frozen_fsbs, axis=0)

plt.figure(figsize=(8, 4))
plt.bar(np.arange(p) - 0.2 , frozen_frac, alpha=0.8, label="Sticky")
plt.bar(np.arange(p) + 0.2, partial_frozen_frac, alpha=0.8, label="Factorized Sticky")
plt.xticks(np.arange(p), [f"$\\beta_{j}$" for j in range(p)])
plt.ylabel("Fraction frozen")
plt.title("Sticky sampler: fraction of time each coefficient is frozen")
plt.legend()
plt.tight_layout()
plt.show()

print("Sticky frozen fractions:")
print(np.round(frozen_frac, 3))
print("Factorized Sticky frozen fractions:")
print(np.round(partial_frozen_frac, 3))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = t_sbs
x2 = samp_sbs[:, 2]
frozen2 = frozen_sbs[:, 2]

plt.figure(figsize=(10,4))
plt.plot(t_sbs, x2, label="Sticky sample path")
plt.axhline(0.0, color="black", linestyle="--", alpha=0.5)

in_block = False
start = None
for i in range(len(t_sbs)):
    if frozen2[i] and not in_block:
        start = t_sbs[i]
        in_block = True
    elif not frozen2[i] and in_block:
        plt.axvspan(start, t_sbs[i], color="gray", alpha=0.25)
        in_block = False

if in_block:
    plt.axvspan(start, t_sbs[-1], color="gray", alpha=0.25, label="Frozen")

plt.xlabel("t")
plt.ylabel(f"beta[{k}]")
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = t_fsbs
x2 = samp_fsbs[:, 2]
frozen2 = frozen_fsbs[:, 2]

plt.figure(figsize=(10,4))
plt.plot(t_fsbs, x2, label="Factorized Sticky sample path")
plt.axhline(0.0, color="black", linestyle="--", alpha=0.5)

in_block = False
start = None
for i in range(len(t_fsbs)):
    if frozen2[i] and not in_block:
        start = t_fsbs[i]
        in_block = True
    elif not frozen2[i] and in_block:
        plt.axvspan(start, t_fsbs[i], color="gray", alpha=0.25)
        in_block = False

if in_block:
    plt.axvspan(start, t_fsbs[-1], color="gray", alpha=0.25, label="Frozen")

plt.xlabel("t")
plt.ylabel(f"beta[{k}]")
plt.legend()
plt.show()

## Logistic regression

In [16]:
# ============================================================

def simulate_logistic_regression(n=300, p=5, seed=1):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, p))
    beta_true = np.array([2.0, -1.5, 0.5, 1.0, -1.0])[:p]
    if len(beta_true) < p:
        beta_true = np.concatenate([beta_true, np.zeros(p - len(beta_true))])

    logits = X @ beta_true
    probs = 1.0 / (1.0 + np.exp(-logits))
    y = rng.binomial(1, probs, size=n)
    return X, y, beta_true


# ============================================================
# Negative log posterior targets E(beta)
# ============================================================

def make_E_logistic_regression(X, y, tau=2.0):
    """
    Bayesian logistic regression with:
        y_i | beta ~ Bernoulli(sigmoid(x_i^T beta))
        beta ~ N(0, tau^2 I)

    Returns E(beta) = negative log posterior up to additive constant.
    """
    X = anp.array(X)
    y = anp.array(y)

    def E(beta):
        eta = X @ beta
        # stable log-likelihood:
        # - sum [ y*eta - log(1+exp(eta)) ]
        nll = anp.sum(anp.logaddexp(0.0, eta) - y * eta)
        nprior = 0.5 / tau**2 * anp.sum(beta**2)
        return nll + nprior

    return E


In [17]:
# Linear regression
X_log_reg, y_log_reg, beta_true_log_reg = simulate_logistic_regression(n=200, p=10, seed=1)
E_log_reg = make_E_logistic_regression(X_log_reg, y_log_reg, tau=2.0)

In [18]:
sampler_log_reg = AutomaticBoomerangSampler(E_log_reg, dim=X_log_reg.shape[1])
sampler_partial_log_reg = FactorizedAutomaticBoomerangSampler(E_log_reg, dim=X_log_reg.shape[1])
sampler_sticky_log_reg = StickyAutomaticBoomerangSampler(E_log_reg, dim=X_log_reg.shape[1], kappa=1)
sampler_partial_sticky_log_reg = FactorizedStickyAutomaticBoomerangSampler(E_log_reg, dim=X_log_reg.shape[1], kappa=1)


In [19]:
import numpy as np

sampler_log_reg.preprocess(
    method="laplace",
    # x_ref=x_ref_right,
    # Sigma_inv=Sigma_inv_well
)

sampler_partial_log_reg.preprocess(
    method="diagonal",
    # x_ref=x_ref_right,
    # Sigma_inv=Sigma_inv_well
)

sampler_sticky_log_reg.preprocess(
    method="laplace",
    # x_ref=x_ref_right,
    # Sigma_inv=Sigma_inv_well
)

sampler_partial_sticky_log_reg.preprocess(
    method="diagonal",
    # x_ref=x_ref_right,
    # Sigma_inv=Sigma_inv_well
)

In [ ]:
log_reg_out_auto_bs = sampler_log_reg.sample_auto(T=2000, adapt_t_max=True, refine=True, safety=1.10, n_grid=10)
log_reg_t_auto_bs = log_reg_out_auto_bs["t"]
log_reg_x_auto_bs = log_reg_out_auto_bs["x"]

In [ ]:
log_reg_out_auto_fbs = sampler_partial_log_reg.sample_auto(T=2000, adapt_t_max=True, refine=True, safety=1.10, n_grid=10)
log_reg_t_auto_fbs = log_reg_out_auto_fbs["t"]
log_reg_x_auto_fbs = log_reg_out_auto_fbs["x"]

In [ ]:
log_reg_out_auto_sticky_bs = sampler_sticky_log_reg.sample_auto(T=2000, adapt_t_max=True, refine=True, safety=1.10, n_grid=10)
log_reg_t_auto_sticky_bs = log_reg_out_auto_sticky_bs["t"]
log_reg_x_auto_sticky_bs = log_reg_out_auto_sticky_bs["x"]

In [ ]:
log_reg_out_auto_factorized_sticky_bs = sampler_partial_sticky_log_reg.sample_auto(T=2000, adapt_t_max=True, refine=True, safety=1.10, n_grid=10)
log_reg_t_auto_factorized_sticky_bs = log_reg_out_auto_factorized_sticky_bs["t"]
log_reg_x_auto_factorized_sticky_bs = log_reg_out_auto_factorized_sticky_bs["x"]

In [24]:
# ------------------------------------------------------------
# Uniform-in-time samples
# ------------------------------------------------------------
t_bs, samp_bs = resample_pdmp_path(log_reg_out_auto_bs, sampler_log_reg, n_samples=4000, sticky=False)
t_fbs, samp_fbs = resample_pdmp_path(log_reg_out_auto_fbs, sampler_partial_log_reg, n_samples=4000, sticky=False)
t_sbs, samp_sbs, frozen_sbs = resample_pdmp_path(log_reg_out_auto_sticky_bs, sampler_sticky_log_reg, n_samples=4000, sticky=True)
t_fsbs, samp_fsbs, frozen_fsbs = resample_pdmp_path(log_reg_out_auto_factorized_sticky_bs, sampler_partial_sticky_log_reg, n_samples=4000, sticky=True)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1) Posterior summaries
# ------------------------------------------------------------
def posterior_summary(samples):
    mean = np.mean(samples, axis=0)
    q05 = np.quantile(samples, 0.05, axis=0)
    q95 = np.quantile(samples, 0.95, axis=0)
    return mean, q05, q95

mean_bs, q05_bs, q95_bs = posterior_summary(samp_bs)
mean_fbs, q05_fbs, q95_fbs = posterior_summary(samp_fbs)
mean_sbs, q05_sbs, q95_sbs = posterior_summary(samp_sbs)
mean_fsbs, q05_fsbs, q95_fsbs = posterior_summary(samp_fsbs)

p = samp_bs.shape[1]
idx = np.arange(p)

plt.figure(figsize=(10, 5))
plt.errorbar(idx - 0.3, mean_bs, 
             yerr=[mean_bs - q05_bs, q95_bs - mean_bs],
             fmt='o', capsize=4, label='Boomerang')
plt.errorbar(idx - 0.1, mean_fbs, 
             yerr=[mean_fbs - q05_fbs, q95_fbs - mean_fbs],
             fmt='o', capsize=4, label='Factorized')
plt.errorbar(idx + 0.1, mean_sbs, 
             yerr=np.maximum(0.0, np.min([mean_sbs - q05_sbs, q95_sbs - mean_sbs])),
             fmt='o', capsize=4, label='Sticky')
plt.errorbar(idx + 0.3, mean_fsbs, 
             yerr=np.maximum(0.0, np.min([mean_fsbs - q05_fsbs, q95_fsbs - mean_fsbs])),
             fmt='o', capsize=4, label='Factorized Sticky')
plt.scatter(idx, beta_true_log_reg, marker='x', s=80, color='black', label='True')
plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.xticks(idx, [f"$\\beta_{j}$" for j in range(p)])
plt.ylabel("Coefficient value")
plt.title("Posterior means and 90% intervals")
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 2) Trace + histogram for selected coefficients
# ------------------------------------------------------------
sel = np.arange(min(3, p))   # first 3 coefficients

fig, axes = plt.subplots(len(sel), 2, figsize=(12, 3 * len(sel)))
if len(sel) == 1:
    axes = np.array([axes])

for r, k in enumerate(sel):
    # trace
    axes[r, 0].plot(t_bs, samp_bs[:, k], lw=1, alpha=0.8, label="Boomerang")
    axes[r, 0].plot(t_fbs, samp_fbs[:, k], lw=1, alpha=0.8, label="Factorized")
    axes[r, 0].plot(t_sbs, samp_sbs[:, k], lw=1, alpha=0.8, label="Sticky")
    axes[r, 0].plot(t_fsbs, samp_fsbs[:, k], lw=1, alpha=0.8, label="Factorized Sticky")
    axes[r, 0].axhline(beta_true_log_reg[k], color="black", linestyle="--", alpha=0.7)
    axes[r, 0].set_title(f"Trace: beta[{k}]")
    axes[r, 0].set_xlabel("t")
    axes[r, 0].set_ylabel(f"beta[{k}]")

    # histogram
    axes[r, 1].hist(samp_bs[:, k], bins=40, density=True, alpha=0.4, label="Boomerang")
    axes[r, 1].hist(samp_fbs[:, k], bins=40, density=True, alpha=0.4, label="Factorized")
    axes[r, 1].hist(samp_sbs[:, k], bins=40, density=True, alpha=0.4, label="Sticky")
    axes[r, 1].hist(samp_fsbs[:, k], bins=40, density=True, alpha=0.4, label="Factorized Sticky")
    axes[r, 1].axvline(beta_true_log_reg[k], color="black", linestyle="--", alpha=0.7)
    axes[r, 1].set_title(f"Marginal: beta[{k}]")
    axes[r, 1].set_xlabel(f"beta[{k}]")

axes[0, 0].legend()
axes[0, 1].legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 3) Sticky-specific: fraction of time frozen
# ------------------------------------------------------------
frozen_frac = np.mean(frozen_sbs, axis=0)
partial_frozen_frac = np.mean(frozen_fsbs, axis=0)

plt.figure(figsize=(8, 4))
plt.bar(np.arange(p) - 0.1, frozen_frac, alpha=0.8, label="Sticky")
plt.bar(np.arange(p) + 0.1, partial_frozen_frac, alpha=0.8, label="Factorized Sticky")
plt.xticks(np.arange(p), [f"$\\beta_{j}$" for j in range(p)])
plt.ylabel("Fraction frozen")
plt.title("Sticky sampler: fraction of time each coefficient is frozen")
plt.tight_layout()
plt.show()

print("Sticky frozen fractions:")
print(np.round(frozen_frac, 3))

print("Factorized Sticky frozen fractions:")
print(np.round(partial_frozen_frac, 3))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = t_sbs
x2 = samp_sbs[:, 2]
frozen2 = frozen_sbs[:, 2]

plt.figure(figsize=(10,4))
plt.plot(t_sbs, x2, label="Sticky sample path")
plt.axhline(0.0, color="black", linestyle="--", alpha=0.5)

in_block = False
start = None
for i in range(len(t_sbs)):
    if frozen2[i] and not in_block:
        start = t_sbs[i]
        in_block = True
    elif not frozen2[i] and in_block:
        plt.axvspan(start, t_sbs[i], color="gray", alpha=0.25)
        in_block = False

if in_block:
    plt.axvspan(start, t_sbs[-1], color="gray", alpha=0.25, label="Frozen")

plt.xlabel("t")
plt.ylabel(f"beta[{k}]")
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = t_fsbs
x2 = samp_fsbs[:, 2]
frozen2 = frozen_fsbs[:, 2]

plt.figure(figsize=(10,4))
plt.plot(t_fsbs, x2, label="Factorized Sticky sample path")
plt.axhline(0.0, color="black", linestyle="--", alpha=0.5)

in_block = False
start = None
for i in range(len(t_fsbs)):
    if frozen2[i] and not in_block:
        start = t_fsbs[i]
        in_block = True
    elif not frozen2[i] and in_block:
        plt.axvspan(start, t_fsbs[i], color="gray", alpha=0.25)
        in_block = False

if in_block:
    plt.axvspan(start, t_fsbs[-1], color="gray", alpha=0.25, label="Frozen")

plt.xlabel("t")
plt.ylabel(f"beta[{k}]")
plt.legend()
plt.show()